In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached symengine-0.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.8 MB)
Using cached symengine-0.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.6 MB)
  Attempting uninstall: qiskit━━━━━━━━━━━━━━━━━━ 0/2 [symengine]
    Found existing installation: qiskit 2.3.0 0/2 [symengine]
    Uninstalling qiskit-2.3.0:━━━━━━━━━━━━━━ 0/2 [symengine]
      Successfully uninstalled qiskit-2.3.0━ 0/2 [symengine]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [qiskit]2m1/2 [qiskit]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /home/jovyan/.qbraid/environments/69ls/pyenv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached qiskit_aer-0.15.1-cp312-cp3

In [3]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [4]:
# Useful constants
root2 = math.sqrt(2)

denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2)

W_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                       [  1 / denom2 , (root2 - 1) / denom2 ] ]
W_transform = Operator(W_transform_matrix)

V_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                       [ -1 / denom2 , (root2 - 1) / denom2 ] ]
V_transform = Operator(V_transform_matrix)

# Construct a circuit that prepares the state 1/sqrt(2) ( |01> - |10> )
def entangledPair():
    q = QuantumCircuit(2)
    q.x(1)
    q.h(0)
    q.cx(0,1)
    q.z(0)
    return q

backend = BasicSimulator()

first_choice_circuit = QuantumCircuit(1,1)
first_choice_circuit.ry(2 * math.acos(math.sqrt(1/3)), 0)
first_choice_circuit.measure(0,0)
compiled_first_choice_circuit = transpile(first_choice_circuit, backend)

second_choice_circuit = QuantumCircuit(1,1)
second_choice_circuit.h(0)
second_choice_circuit.measure(0,0)
compiled_second_choice_circuit = transpile(second_choice_circuit, backend)


def singleBitResult(compiled_circuit):
    job_sim = backend.run(compiled_circuit, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled_circuit)
    return int(list(counts.keys())[0])


def randomChoice():
    first_bit = singleBitResult(compiled_first_choice_circuit)
    if first_bit == 0:
        return 1
    second_bit = singleBitResult(compiled_second_choice_circuit)
    if second_bit == 0:
        return 2
    return 3


# Protocol circuit logic for Bob and Alice step 4/5
def measurementCircuit(q, alice_choice, bob_choice):

    if bob_choice == 1:
        # B1 = W
        q.unitary(W_transform_matrix, [1])
    elif bob_choice == 2:
        # B2 = Z
        pass
    elif bob_choice == 3:
        # B3 = V
        q.unitary(V_transform_matrix, [1])

    if alice_choice == 1:
        # A1 = X
        q.h(0)
    elif alice_choice == 2:
        # A2 = W
        q.unitary(W_transform_matrix, [0])
    elif alice_choice == 3:
        # A3 = Z
        pass

    q.measure_all()
    return q


# Actually run/setup circuit
def protocolRound():
    # step 1
    q = entangledPair() # assume shared

    # attacker intercepts before bob/alice receive
    q = eveIntercepts(q)

    # step 2/3
    alice_choice, bob_choice = randomChoice(), randomChoice()

    # step 4/5 (both happen in same circuit for simplicity)
    compiled_measurement_circuit = transpile(measurementCircuit(q, alice_choice, bob_choice), backend)

    job_sim = backend.run(compiled_measurement_circuit, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled_measurement_circuit)
    bit_string = list(counts.keys())[0]

    bob_bit = int(bit_string[0])
    alice_bit = int(bit_string[1])

    return alice_bit, bob_bit, alice_choice, bob_choice

def plusMinusOne(bit):
    if bit == 0:
        return 1
    return -1

# Eve intercepts qubits, measures them and resends according to protocol
def eveIntercepts(q):
    eve_circuit = q.copy()
    eve_circuit.measure_all()
    compiled_eve_circuit = transpile(eve_circuit, backend)

    job_sim = backend.run(compiled_eve_circuit, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled_eve_circuit)
    bit_string = list(counts.keys())[0]

    bob_eve_bit = int(bit_string[0])
    alice_eve_bit = int(bit_string[1])

    resent_q = QuantumCircuit(2)
    if alice_eve_bit == 1:
        resent_q.x(0)
    if bob_eve_bit == 1:
        resent_q.x(1)
    return resent_q

In [6]:
N = 100 # key size
rounds = math.ceil(9 * N / 2)

aliceData, bobData = [], []
for round_number in range(rounds):

    alice_bit, bob_bit, alice_choice, bob_choice = protocolRound()

    aliceData.append((alice_bit, alice_choice))
    bobData.append((bob_bit, bob_choice))

alice_key = []
bob_key = []

XW_results = []
XV_results = []
ZW_results = []
ZV_results = []

for alice_data, bob_data in zip(aliceData, bobData):
    alice_bit = alice_data[0]
    bob_bit = bob_data[0]
    alice_choice = alice_data[1]
    bob_choice = bob_data[1]

    # Key cases
    if alice_choice == 2 and bob_choice == 1:
        alice_key.append(alice_bit)
        bob_key.append(1 - bob_bit)

    elif alice_choice == 3 and bob_choice == 2:
        alice_key.append(alice_bit)
        bob_key.append(1 - bob_bit)

    # Bell test cases
    elif alice_choice == 1 and bob_choice == 1:
        XW_results.append(plusMinusOne(alice_bit) * plusMinusOne(bob_bit))

    elif alice_choice == 1 and bob_choice == 3:
        XV_results.append(plusMinusOne(alice_bit) * plusMinusOne(bob_bit))

    elif alice_choice == 3 and bob_choice == 1:
        ZW_results.append(plusMinusOne(alice_bit) * plusMinusOne(bob_bit))

    elif alice_choice == 3 and bob_choice == 3:
        ZV_results.append(plusMinusOne(alice_bit) * plusMinusOne(bob_bit))

    # else discard
    else:
        pass

XW_average = sum(XW_results) / len(XW_results)
XV_average = sum(XV_results) / len(XV_results)
ZW_average = sum(ZW_results) / len(ZW_results)
ZV_average = sum(ZV_results) / len(ZV_results)

S = abs(XW_average - XV_average + ZW_average + ZV_average)

print("keys:")
print(alice_key[:20])
print(bob_key[:20])
print("Matches:", alice_key == bob_key)

print()
print("S =", S)
print("2*sqrt(2) =", 2 * math.sqrt(2))

print()
print("keys dont match and S <= 2 so entanglement is broken, intruder is detected, key exchange attempt can be safely abandonded")
print("protocol functions correctly")


keys:
[1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1]
[1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1]
Matches: False

S = 1.7544863444559493
2*sqrt(2) = 2.8284271247461903

keys dont match and S <= 2 so entanglement is broken, intruder is detected, key exchange attempt can be safely abandonded
protocol functions correctly
